In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef


df = pd.read_csv("/kaggle/input/nli-training/train.csv")

# Combine premise and hypothesis
df["text_pair"] = df["premise"] + " [SEP] " + df["hypothesis"]

# Split features and labels
X = df["text_pair"]
y = df["label"]


In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=5000,
    ngram_range=(1, 2),
)

X_tfidf = vectorizer.fit_transform(X)

In [ ]:
# Split into train and dev sets
X_train, X_dev, y_train, y_dev = train_test_split(X_tfidf, y, test_size=0.2)

In [ ]:
# Train an SVM classifier
clf = SVC(kernel='linear', C=1.0)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_dev)

In [4]:
print("Classification Report:\n", classification_report(y_dev, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_dev, y_pred))

# Custom metric summary
print(f"Accuracy: {accuracy_score(y_dev, y_pred):.4f}")
print(f"Macro-F1: {f1_score(y_dev, y_pred, average='macro'):.4f}")
print(f"Macro-Precision: {precision_score(y_dev, y_pred, average='macro'):.4f}")
print(f"Macro-Recall: {recall_score(y_dev, y_pred, average='macro'):.4f}")
print(f"MCC: {matthews_corrcoef(y_dev, y_pred):.4f}")


Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.54      0.56      2335
           1       0.61      0.65      0.63      2552

    accuracy                           0.60      4887
   macro avg       0.60      0.59      0.59      4887
weighted avg       0.60      0.60      0.60      4887

Confusion Matrix:
 [[1269 1066]
 [ 904 1648]]
Accuracy: 0.5969
Macro-F1: 0.5945
Macro-Precision: 0.5956
Macro-Recall: 0.5946
MCC: 0.1902


In [5]:
# # Prepare test text the same way
df_test = pd.read_csv("/kaggle/input/nli-development-dataset/dev.csv")  # test data with premise/hypothesis only
df_test["text_pair"] = df_test["premise"] + " [SEP] " + df_test["hypothesis"]
X_test = vectorizer.transform(df_test["text_pair"])

# Predict on test data
test_preds = clf.predict(X_test)

# Save to CSV
output = pd.DataFrame({"prediction": test_preds})
output.to_csv("Group_52_A.csv", index=False)
